# Pipeline Big Data — Monitoreo de Deforestación en el Chocó

**Proyecto Final · Módulo Big Data**
**Integrantes:** Dahiana Andrea Moreno Palomeque

Este notebook implementa el pipeline real de ingesta y transformación de datos para el
proyecto de monitoreo de deforestación en el departamento del Chocó, siguiendo la
**estrategia Medallion (Bronze → Silver → Gold)** sobre **Databricks Free Edition**
(Unity Catalog + Lakeflow).

**Antes de ejecutar:**
1. Sube el archivo `AREAS_DEFORESTADAS_CHOCO_20260511.csv` a un **Volume** de Unity Catalog
   (Catalog Explorer → tu catálogo → tu esquema → Volumes → Create Volume → Upload).
2. Ajusta las variables `catalog`, `schema` y `volume_path` en la celda siguiente para que
   coincidan con la ubicación donde subiste el archivo.
3. Ejecuta las celdas en orden (Run All).

In [ ]:
# ============================================================
# CONFIGURACIÓN — ajusta estos valores a tu workspace
# ============================================================
catalog = "workspace"                 # catálogo de Unity Catalog (revisa Catalog Explorer)
schema = "deforestacion_choco"        # esquema donde se crearán las tablas Bronze/Silver/Gold
volume_path = "/Volumes/workspace/default/deforestacion_choco_raw"  # ruta del Volume donde subiste el CSV
csv_filename = "AREAS_DEFORESTADAS_CHOCO_20260511.csv"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
print(f"Esquema listo: {catalog}.{schema}")

## 1. Capa Bronze — Ingesta cruda

Se realiza la carga inicial del archivo CSV **sin ninguna transformación**, preservando
exactamente el formato original (coordenadas en texto GMS, área con separador decimal de
coma). Se agregan únicamente columnas de metadatos de ingesta para trazabilidad.

In [ ]:
from pyspark.sql import functions as F

raw_path = f"{volume_path}/{csv_filename}"

bronze_df = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .option("encoding", "ISO-8859-1")
    .csv(raw_path)
    .withColumnRenamed("TIPO GEOMETRIA", "TIPO_GEOMETRIA")
    .withColumn("fecha_ingesta", F.current_timestamp())
    .withColumn("archivo_origen", F.lit(csv_filename))
)

bronze_table = f"{catalog}.{schema}.bronze_deforestacion_choco"
bronze_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"Tabla Bronze creada: {bronze_table}")
print(f"Registros cargados: {bronze_df.count()}")
display(bronze_df.limit(10))

## 2. Capa Silver — Limpieza, estandarización y enriquecimiento

Se aplican las transformaciones necesarias para que los datos sean confiables y consistentes:

- Conversión de coordenadas de Grados-Minutos-Segundos (texto) a grados decimales (WGS84).
- Conversión de `AREA_Ha` de texto con coma decimal a tipo numérico.
- Estandarización de nombres de municipio.
- Marca de calidad para el sesgo técnico de nubosidad detectado en 2016.
- Eliminación de duplicados y validación de rangos lógicos.

In [ ]:
from pyspark.sql.types import DoubleType
import re

def gms_a_decimal(gms):
    """Convierte coordenadas en formato Grados-Minutos-Segundos a grados decimales."""
    if gms is None:
        return None
    partes = re.findall(r"[\d.,]+", gms)
    if len(partes) < 3:
        return None
    grados, minutos, segundos = [float(v.replace(",", ".")) for v in partes[:3]]
    decimal = grados + minutos / 60 + segundos / 3600
    if "S" in gms.upper() or "W" in gms.upper():
        decimal *= -1
    return round(decimal, 6)

gms_udf = F.udf(gms_a_decimal, DoubleType())

bronze = spark.table(bronze_table)

silver_df = (
    bronze
    .withColumn("area_ha", F.regexp_replace(F.col("AREA_Ha"), ",", ".").cast(DoubleType()))
    .withColumn("latitud_dec", gms_udf(F.col("LATITUD")))
    .withColumn("longitud_dec", gms_udf(F.col("LONGITUD")))
    .withColumn("municipio_std", F.upper(F.trim(F.col("MUNICIPIO"))))
    .withColumn("anio", F.col("AÑO").cast("int"))
    .withColumn(
        "alerta_calidad_2016",
        F.when(
            (F.col("anio") == 2016) & F.col("OBSERVACION").contains("nubosidad"),
            F.lit(True),
        ).otherwise(F.lit(False)),
    )
    .dropDuplicates(["ID"])
    .filter((F.col("area_ha") > 0) & F.col("latitud_dec").isNotNull())
)

silver_table = f"{catalog}.{schema}.silver_deforestacion_choco_limpio"
silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

print(f"Tabla Silver creada: {silver_table}")
print(f"Registros tras limpieza: {silver_df.count()}")
display(silver_df.limit(10))

## 3. Capa Gold — Agregados listos para negocio

Se construye la tabla agregada que alimenta el tablero de monitoreo y el modelo predictivo:
hectáreas y eventos totales por municipio-año, causa predominante y nivel de alerta.

In [ ]:
gold_df = (
    spark.table(silver_table)
    .groupBy("municipio_std", "anio")
    .agg(
        F.sum("area_ha").alias("hectareas_totales"),
        F.count("ID").alias("eventos_totales"),
        F.first("CAUSA").alias("causa_predominante"),
    )
    .withColumn(
        "nivel_alerta",
        F.when(F.col("hectareas_totales") > 200, "Extrema")
        .when(F.col("hectareas_totales") > 50, "Alta")
        .when(F.col("hectareas_totales") > 10, "Media")
        .otherwise("Moderada"),
    )
)

gold_table = f"{catalog}.{schema}.gold_hotspots_deforestacion"
gold_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

print(f"Tabla Gold creada: {gold_table}")
display(gold_df.orderBy(F.desc("hectareas_totales")).limit(15))

## 4. Análisis Descriptivo

Resúmenes estadísticos sobre la capa Silver, base de las Figuras 1, 2 y 3 del informe.

In [ ]:
silver = spark.table(silver_table)

print("--- Eventos por año (revisa la caída de 2016 por nubosidad) ---")
display(silver.groupBy("anio").count().orderBy("anio"))

print("--- Top municipios por hectáreas deforestadas acumuladas ---")
display(
    silver.groupBy("municipio_std")
    .agg(F.sum("area_ha").alias("ha_total"))
    .orderBy(F.desc("ha_total"))
    .limit(10)
)

print("--- Área deforestada por causa ---")
display(
    silver.groupBy("CAUSA")
    .agg(F.sum("area_ha").alias("ha_total"), F.count("ID").alias("eventos"))
    .orderBy(F.desc("ha_total"))
)

## 5. Modelado Predictivo con seguimiento en MLflow

Se entrena un modelo de clasificación (Random Forest) que estima la probabilidad de que una
combinación municipio-año caiga en nivel de alerta **Alta** o **Extrema**, a partir de la
causa predominante y el número de eventos registrados. El experimento se registra en MLflow
para trazabilidad y reproducibilidad.

In [ ]:
import os
import shutil
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# NOTA: usamos MlflowClient de bajo nivel (en vez de mlflow.start_run()) y fijamos
# explícitamente tracking_uri/registry_uri a almacenamiento local. Esto evita un error
# conocido en algunos workspaces de Databricks Free Edition donde mlflow.start_run()
# intenta resolver automáticamente el Model Registry de Unity Catalog mediante una
# consulta a spark.conf que falla con CONFIG_NOT_AVAILABLE_WITHOUT_UC_SHARING_CATALOG.
LOCAL_TRACKING_URI = "file:/tmp/mlruns_deforestacion_choco"
mlflow.set_tracking_uri(LOCAL_TRACKING_URI)
mlflow.set_registry_uri(LOCAL_TRACKING_URI)
client = MlflowClient(tracking_uri=LOCAL_TRACKING_URI, registry_uri=LOCAL_TRACKING_URI)

gold_pd = spark.table(gold_table).toPandas()
gold_pd["alerta_alta"] = gold_pd["nivel_alerta"].isin(["Alta", "Extrema"]).astype(int)

features = pd.get_dummies(
    gold_pd[["causa_predominante", "eventos_totales"]], columns=["causa_predominante"]
)
X = features
y = gold_pd["alerta_alta"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

experiment_name = "riesgo_deforestacion_choco"
exp = client.get_experiment_by_name(experiment_name)
experiment_id = exp.experiment_id if exp else client.create_experiment(experiment_name)

run = client.create_run(experiment_id=experiment_id, run_name="riesgo_deforestacion_rf_v1")
run_id = run.info.run_id

modelo = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
modelo.fit(X_train, y_train)

proba = modelo.predict_proba(X_test)[:, 1]
pred = modelo.predict(X_test)
auc = roc_auc_score(y_test, proba)
f1 = f1_score(y_test, pred)

client.log_param(run_id, "n_estimators", 200)
client.log_param(run_id, "max_depth", 6)
client.log_metric(run_id, "auc_roc", auc)
client.log_metric(run_id, "f1_score", f1)

try:
    local_model_dir = "/tmp/modelo_riesgo_deforestacion"
    if os.path.exists(local_model_dir):
        shutil.rmtree(local_model_dir)
    mlflow.sklearn.save_model(modelo, path=local_model_dir)
    client.log_artifacts(run_id, local_model_dir, artifact_path="modelo_riesgo_deforestacion")
except Exception as e:
    print(f"Aviso: no se pudo guardar el artefacto del modelo ({type(e).__name__}).")

client.set_terminated(run_id)

print(f"MLflow run_id: {run_id}")
print(f"AUC-ROC: {auc:.3f}")
print(f"F1-score: {f1:.3f}")
print()
print(classification_report(y_test, pred, target_names=["Baja/Moderada", "Alta/Extrema"]))

## 6. Próximos pasos para completar la evidencia del proyecto

1. **Job (Lakeflow Jobs):** ve a *Workflows → Jobs & Pipelines → Create Job*, agrega este
   notebook como una tarea y ejecútalo. Toma una captura de la ejecución exitosa para
   `images/JOB.png`.
2. **Lineage Graph (Unity Catalog):** ve a *Catalog Explorer*, busca
   `gold_hotspots_deforestacion`, abre la pestaña **Lineage** y toma una captura del grafo
   Bronze → Silver → Gold para `images/Lineage.png`.
3. **Ejecución del notebook:** toma una captura de esta ejecución completa (con los
   resultados visibles) para `images/Databricks.png`.
4. **MLflow:** en el panel de *Experiments*, abre la corrida `riesgo_deforestacion_rf_v1`
   y revisa las métricas registradas (opcional, captura adicional).